In [1]:
# --- IMPORTS ---

import os
import pandas as pd
import matplotlib.pyplot as plt
import re

In [2]:
# --- CONFIGURATION ---

TELEMETRY_BASE_DIR = 'Telemetry/Real'
PLOTS_OUTPUT_DIR = 'Plots/Real'

Y_AXIS_COLUMNS = [
    'Time',
    'car_speed',
    'STEER',
    'P_F_BRAKE',
    'PPS.proportion',
    'flDamper_Corr',
    'frDamper_Corr',
    'rlDamper_Corr',
    'rrDamper_Corr'
]

LAP_COLOURS = [
    'blue', 'red', 'green', 'orange', 'purple',
    'cyan', 'magenta', 'brown', 'pink', 'gray'
]


In [3]:
# --- MAIN SCRIPT ---

def create_plots():

    #--- 1. Set up directories and overwriting ---
    
    if not os.path.exists(PLOTS_OUTPUT_DIR):
        print(f"Creating output directory: {PLOTS_OUTPUT_DIR}")
        os.makedirs(PLOTS_OUTPUT_DIR)

    overwrite_all = None
    while overwrite_all is None:
        response = input("Do you want to overwrite existing plot files? (y/n): ").lower()
        if response in ['y', 'yes']:
            overwrite_all = True
        else:
            overwrite_all = False

    
    #--- 2. Iterate through each track folder ---

    if not os.path.isdir(TELEMETRY_BASE_DIR):
        print(f"Error: Telemetry directory not found at '{TELEMETRY_BASE_DIR}'")
        return
    
    track_names = [d for d in os.listdir(TELEMETRY_BASE_DIR) if os.path.isdir(os.path.join(TELEMETRY_BASE_DIR, d))]

    for track_name in track_names:
        track_path = os.path.join(TELEMETRY_BASE_DIR, track_name)
        print(f"\nProcessing track: {track_name}")

        output_filename = os.path.join(PLOTS_OUTPUT_DIR, f"plots_{track_name}.png")

        if not overwrite_all and os.path.exists(output_filename):
            print(f"Skipping '{track_name}' as plot already exists.")
            continue

        try:
            telemetry_files = sorted([f for f in os.listdir(track_path) if f.startswith('lap') and f.endswith('.csv')])
            if not telemetry_files:
                print(f"   No telemetry files found for track '{track_name}'. Skipping.")
                continue
        except FileNotFoundError:
            print(f"   Directory not found: {track_path}. Skipping.")
            continue


        #--- 3. Pre-load all lap data for the track ---

        lap_data_list = []
        for file_name in telemetry_files:
            file_path = os.path.join(track_path, file_name)
            try:
                # Extract lap number from the filename for the legend
                lap_match = re.search(r'lap(\d+)', file_name)
                lap_label = f"Lap {lap_match.group(1)}" if lap_match else file_name

                df = pd.read_csv(file_path)

                # Normalise distance and time to start at 0
                df['Distance'] = df['Distance'] - df['Distance'].iloc[0]
                df['Time'] = df['Time'] - df['Time'].iloc[0]
                lap_data_list.append({'df': df, 'label': lap_label})

            except Exception as e:
                print(f"Error processing file '{file_name}': {e}. Skipping this file.")

        if not lap_data_list:
            print(f"   No data could be loaded for track '{track_name}'. Skipping plot generation.")
            continue


        #--- 4. Create the plot with vertical subplots ---

        num_plots = len(Y_AXIS_COLUMNS)
        fig, axs = plt.subplots(
            nrows = num_plots,
            ncols = 1,
            figsize = (15, 3 * num_plots),
            sharex = True
        )

        # Ensure 'axs' is a list
        if num_plots == 1:
            axs = [axs]
        
        for plot_index, column in enumerate(Y_AXIS_COLUMNS):
            current_ax = axs[plot_index]
            for lap_index, lap_data in enumerate(lap_data_list):
                lap_colour = LAP_COLOURS[lap_index % len(LAP_COLOURS)]
                df = lap_data['df']
                if column in df.columns:
                    current_ax.plot(df['Distance'], df[column], color=lap_colour)
                else:
                    print(f"   Warning: Column '{column}' not found for {lap_data['label']}")

            current_ax.set_ylabel(column)
            current_ax.grid(True)


        #--- 5. Finalise and save plot ---

        legend_handles = []
        for i in range(len(lap_data_list)):
            label = lap_data_list[i]['label']
            colour = LAP_COLOURS[i % len(LAP_COLOURS)]
            legend_handles.append(plt.Line2D([0], [0], color=colour, lw=4, label=label))

        fig.legend(handles=legend_handles, title='Laps', bbox_to_anchor = (1.01, 0.95), loc='upper left')

        fig.suptitle(f"Telemetry Data for {track_name}", fontsize = 16)
        
        axs[-1].set_xlabel('Distance (meters)')
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        print(f"   Saving plot to {output_filename}")
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        plt.close(fig)
    
    print("\nScript finished.")

if __name__ == '__main__':
    create_plots()


Processing track: silverstone_gp
   Saving plot to Plots/Real\plots_silverstone_gp.png

Processing track: brands_gp
   Saving plot to Plots/Real\plots_brands_gp.png

Processing track: hungaroring
   Saving plot to Plots/Real\plots_hungaroring.png

Processing track: donington_gp
   Saving plot to Plots/Real\plots_donington_gp.png

Processing track: spa
   Saving plot to Plots/Real\plots_spa.png

Processing track: zandvoort
   Saving plot to Plots/Real\plots_zandvoort.png

Processing track: Target_Monza
   No telemetry files found for track 'Target_Monza'. Skipping.

Script finished.
